<a href="https://colab.research.google.com/github/ancantos99/proyectointegrador_ia_grupo8/blob/main/notebooks/04_optimizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1.- CARGA DE DATOS**
- Descarga del Dataset del repositorio de hugginface

In [ ]:
# Clonar el repo del dataset en /content/dataset
!git lfs install > /dev/null 2>&1
!git clone https://huggingface.co/datasets/YashJain/UI-Elements-Detection-Dataset /content/dataset > /dev/null 2>&1
print("✅ Git LFS instalado y el dataset clonado en '/content/dataset' de forma silenciosa.")

✅ Git LFS instalado y el dataset clonado en '/content/dataset' de forma silenciosa.


**2.- Instalación de Librerías y Montar Google Drive**


In [ ]:
#Instarlar librerias
!pip install -U ultralytics wandb > /dev/null 2>&1
print("✅ Ultralytics wandb instalado de forma silenciosa")

✅ Ultralytics wandb instalado de forma silenciosa


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**3.- Definición clara de los hiperparámetros y rangos a explorar**

Para la exploración de hiperparámetros, se utilizó la** aplicación Weights & Biases (W&B)**, una plataforma especializada en la gestión y seguimiento de experimentos de machine learning. **Se ejecutaron 50 corridas experimentales empleando el método de Optimización Bayesiana**, con el objetivo de maximizar la métrica mAP50(B), correspondiente al desempeño del modelo YOLOv8 en la detección de objetos.

A continuación se muestra la configuración de Weights & Biases (W&B) que se utilizó, con la métrica a evaluar y los hiperparámetros con sus rangos


In [ ]:
sweep_config = {
    'method': 'bayes',  # usa optimización bayesiana
    'metric': {
        'name': 'metrics/mAP50(B)',  # métrica objetivo YOLOv8
        'goal': 'maximize'
    },
    'parameters': {
        'batch': {'values': [6]},
        'imgsz': {'values': [(1920,1080)]},
        # 🔹 Optimizador
        'optimizer': {'values': ['SGD', 'Adam', 'AdamW']},
        'momentum': {'min': 0.7, 'max': 0.97},     # para los casos de SGD (acelera el aprendizaje y suaviza las actualizaciones de los pesos del modelo durante el entrenamiento)
        # Aumento de datos (True/False)
        'augment': {'values': [True, False]},
        # Tasa de aprendizaje
        "lr0": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e-1},
        "lrf": {"distribution": "uniform", "min": 0.01, "max": 1.0},
        # Penaliza pesos grandes, como una regularización L2
        'weight_decay': {'min': 0.0001, 'max': 0.01},
    }
}

In [ ]:
#VERIFICAR si W&B está activado (Activarlo) en la configuración global de YOLOv8
from ultralytics.utils import SETTINGS
SETTINGS.update({'wandb': True})
print(SETTINGS["wandb"])

True


In [ ]:
import wandb
# Inicializar el entorno de  Weights & Biases ( se necesita el apikey de WB)
wandb.login()

In [ ]:
# Crear el sweep
sweep_id = wandb.sweep(sweep_config, project='opt03bayes_yolov8')

In [ ]:
import wandb
from ultralytics import YOLO

def train_yolo():
    # Inicializa un run de W&B
    wandb.init()
    config = wandb.config

    # Cargar Modelo YoloV8
    model = YOLO("yolov8l.pt")

    model.train(
        data='/content/dataset/dataset.yaml',
        epochs=50,
        imgsz=config.imgsz,
        batch=config.batch,
        optimizer=config.optimizer,
        lr0=config.lr0,
        lrf=config.lrf,
        momentum=config.momentum,      # será ignorado si usa Adam/AdamW
        weight_decay=config.weight_decay,
        augment=config.augment,
        project="/content/drive/MyDrive/MIA/Entrenamientos/Optimizador/opt03bayes_yolov8",
        name=f"sweep_run_{wandb.run.id}",
        tracker='wandb'
    )

**Enviar a ejecutar las pruebas**

In [ ]:
wandb.agent(sweep_id, function=train_yolo, count=50)

**Link público de las ejecuciones guardadas en Weights & Biases (W&B)**

https://api.wandb.ai/links/ancantos99_martin/slzcfke1